# Modelado Predictivo de la Infracción C03

Se desarrolla un modelo predictivo para la infracción C03 (bloquear calzada o intersección) empleando una metodología iterativa en la que cada nuevo modelo aprende de los resultados obtenidos por los modelos previamente entrenados. El proceso inicia con el modelo Holt-Winters, cuyos parámetros se ajustan con base en los hallazgos del análisis exploratorio y la descomposición STL de la serie temporal. Posteriormente, se incorporan secuencialmente los modelos ARIMA, SARIMA, Dynamic Optimized Theta, Ridge, Lasso, Random Forest, XGBoost, LightGBM y KNN, de modo que cada uno aprovecha el conocimiento acumulado por sus predecesores, con el objetivo de mejorar progresivamente el rendimiento predictivo. Las métricas de evaluación utilizadas son RMSE, MAE, MAPE, SMAPE y MSE. Finalmente, se selecciona el mejor modelo para realizar pronósticos del volumen de infracciones por bloqueo de calzada para el año 2026, cuyos datos aún no han sido publicados por la Alcaldía de Barranquilla a través de su portal de Datos Abiertos.

## Carga de librerías

Se importan las librerías necesarias para el desarrollo de modelos predictivos sobre las series temporales de comparendos electrónicos. Se emplean `pandas` y `numpy` para la manipulación y transformación de datos, `plotly.graph_objects` y `plotly.subplots` para la visualización de resultados, y `statsmodels.tsa` para los modelos estadísticos clásicos (Holt-Winters, ARIMA, SARIMA y STL). Para los enfoques de machine learning, se utilizan `scikit-learn` con `RidgeCV`, `MultiTaskLassoCV`, `RandomForestRegressor`, `KNeighborsRegressor` y `GridSearchCV` para optimización de hiperparámetros, junto con `xgboost` (XGBoost) y `lightgbm` (LightGBM). Adicionalmente, se incorporan `sklearn.model_selection.TimeSeriesSplit` para validación con series temporales, `sklearn.preprocessing.StandardScaler` para estandarización de características, y `sklearn.metrics` para las métricas de evaluación (RMSE, MAE, MAPE, SMAPE y MSE). Se suprimen los mensajes de advertencia para mantener una salida limpia y enfocada en los resultados.

In [1]:
import warnings

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL
from scipy import stats as sp_stats
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import MultiTaskLassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings('ignore')

In [2]:
pio.renderers.default = "notebook_connected"

## Carga del DataFrame

Se carga el archivo CSV que contiene los datos de comparendos electrónicos utilizando la función `read_csv` de la librería pandas, y se almacena en el DataFrame `df_comparendos_electronicos`.

In [3]:
df_comparendos_electronicos = pd.read_csv("C:/Users/david/Documents/seminario_investigativo/comparendos_electronicos.csv")

### Conversión de las Fechas a Formato datetime

Se convierte la columna `fecha_comparendo` al tipo `datetime` utilizando el formato especificado (`'%Y %b %d %I:%M:%S %p'`), normalizando la hora a medianoche con el método `.dt.normalize()` para eliminar la información horaria y trabajar únicamente con fechas, dado que todos los registros contienen la hora de 12:00:00 AM. Posteriormente, se imprime el tipo de dato resultante para verificar la correcta conversión.

In [4]:
df_comparendos_electronicos['fecha_comparendo'] = pd.to_datetime(df_comparendos_electronicos['fecha_comparendo'], format='%Y %b %d %I:%M:%S %p').dt.normalize()

print(df_comparendos_electronicos['fecha_comparendo'].dtype)

datetime64[ns]


### Corrección de Valores Nulos

Se rellenan los valores nulos de la columna `DESC_INFRACCION` con la descripción correspondiente al código C14, obtenida del sitio web oficial del Tránsito del Atlántico. Según esta fuente, la infracción C14 corresponde a **"Transitar por sitios restringidos o en horas prohibidas por la autoridad competente"**. Esta corrección se aplica exclusivamente a los 564 registros que presentaban valores nulos, asociados a las nuevas cámaras tipo Carril Bus implementadas en Barranquilla a partir del 17 de octubre de 2025.

**Fuente:** https://transitodelatlantico.gov.co/valor-de-multas-de-transito/

In [5]:
df_comparendos_electronicos['DESC_INFRACCION'] = df_comparendos_electronicos['DESC_INFRACCION'].fillna("Transitar por sitios restringidos o en horas prohibidas por la autoridad competente")  

print(f"Total de valores nulos en el DataFrame: {df_comparendos_electronicos.isnull().sum().sum()}")

Total de valores nulos en el DataFrame: 0


### Corrección de Cámaras Duplicadas

Se unifican los nombres de las cámaras que presentan dos notaciones diferentes para el mismo punto de control. Esta corrección permite evitar la duplicación de ubicaciones en los análisis y garantizar la consistencia de los datos.

In [6]:
df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 53 CON CALLE 104', 'Camara_y_direccion'] = 'CARRERA 53 ENTRE CALLE 104 Y 106'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 84 CON CARRERA 59', 'Camara_y_direccion'] = 'CALLE 84 ENTRE CARRERA 59 Y 59B'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 45 CON CALLE 53', 'Camara_y_direccion'] = 'CALLE 53 CON CARRERA 45'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 45B CARRERA 14', 'Camara_y_direccion'] = 'CALLE 45B CON CARRERA 14'

### Corrección de Infracciones C02 Detectadas por Cámaras Fijas

Dado que estos registros representan una inconsistencia en la base de datos (las cámaras fijas no están diseñadas para detectar estacionamiento prohibido), se procede a eliminarlos del DataFrame principal para garantizar la consistencia de los análisis posteriores. Posteriormente, se verifica que no queden registros residuales de C02 asociados a cámaras fijas, confirmando la correcta limpieza de los datos.

In [7]:
df_comparendos_electronicos = df_comparendos_electronicos[~((df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & 
                                                             (df_comparendos_electronicos['Tipo Camara'] == 'Fijo'))]

## Preparación de la Serie Temporal Mensual

Se extrae y configura la serie temporal mensual correspondiente a la infracción C03 (bloquear calzada o intersección) a partir del DataFrame de comparendos electrónicos. Para ello, se agrupan los registros por mes y código de infracción, sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por período. La serie resultante se indexa con fechas mensuales, se establece una frecuencia mensual ('MS') y se rellenan los valores faltantes con cero, asegurando una secuencia temporal continua y completa para los modelos predictivos. Se imprime información básica de la serie: número de meses, rango temporal y total de comparendos del período.

In [8]:
df_comparendos_electronicos_copy = df_comparendos_electronicos.copy()
df_comparendos_electronicos_copy['año_mes'] = df_comparendos_electronicos_copy['fecha_comparendo'].dt.to_period('M').astype(str)

def preparar_serie_mensual(df, codigo_infraccion):
    infracciones_por_mes = df.groupby(['año_mes', 'COD_INFRACCION'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    
    serie = infracciones_por_mes[infracciones_por_mes['COD_INFRACCION'] == codigo_infraccion].copy()
    serie = serie.set_index('año_mes')['CANTIDAD_INFRACCIONES']
    
    serie.index = pd.to_datetime(serie.index)
    serie = serie.asfreq('MS')
    serie = serie.fillna(0)
    
    return serie

serie_c03 = preparar_serie_mensual(df_comparendos_electronicos_copy, 'C03')
print(f"Serie C03: {len(serie_c03)} meses")
print(f"Desde: {serie_c03.index.min().strftime('%Y-%m')} hasta: {serie_c03.index.max().strftime('%Y-%m')}")
print(f"Total comparendos: {serie_c03.sum():,.0f}")

Serie C03: 96 meses
Desde: 2018-01 hasta: 2025-12
Total comparendos: 65,230


## División de la Serie Temporal en Conjuntos de Entrenamiento y Prueba

Se particiona la serie temporal mensual de la infracción C03 (bloquear calzada o intersección) en dos conjuntos: entrenamiento y prueba. El conjunto de entrenamiento comprende los meses desde enero de 2018 hasta diciembre de 2024, abarcando 84 meses, mientras que el conjunto de prueba corresponde al año completo de 2025, con 12 meses. Esta división permite entrenar los modelos predictivos con los datos históricos y evaluar su desempeño pronosticando el período más reciente (2025), cuyos valores reales ya están disponibles en la base de datos para calcular las métricas de error. Se imprime información detallada de ambos conjuntos, incluyendo su rango temporal y número de meses.

In [9]:
train_start = '2018-01-01'
train_end = '2024-12-01'
test_start = '2025-01-01'
test_end = '2025-12-01'

train_c03 = serie_c03[train_start:train_end].copy()
test_c03 = serie_c03[test_start:test_end].copy()

print(f"Train: {train_c03.index.min().strftime('%Y-%m')} a {train_c03.index.max().strftime('%Y-%m')} ({len(train_c03)} meses)")
print(f"Test: {test_c03.index.min().strftime('%Y-%m')} a {test_c03.index.max().strftime('%Y-%m')} ({len(test_c03)} meses)")

Train: 2018-01 a 2024-12 (84 meses)
Test: 2025-01 a 2025-12 (12 meses)


## Generación de Ventanas de Validación Cruzada Temporal

Se implementa una función de validación cruzada específica para series temporales, que respeta el orden cronológico de los datos y evita la fuga de información del futuro. La función `time_series_cv_manual` genera un conjunto de ventanas de entrenamiento y validación, donde el tamaño de cada ventana de validación es de 12 meses (un año completo). Se configuran 4 ventanas de validación, lo que significa que el modelo se evalúa con los últimos 4 años de datos disponibles. Para la infracción C03, se imprimen los rangos temporales de cada fold, mostrando el período de entrenamiento y validación correspondiente. Esta metodología permite evaluar el desempeño predictivo de los modelos en diferentes horizontes temporales de manera robusta y realista.

In [10]:
def time_series_cv_manual(serie, n_ventanas=4, horizonte=12):
    ventanas = []
    
    for i in range(n_ventanas, 0, -1):
        train_end_idx = len(serie) - (i * horizonte)
        train_fold = serie.iloc[:train_end_idx]
        val_fold = serie.iloc[train_end_idx:train_end_idx + horizonte]
        
        ventanas.append((train_fold, val_fold))
        
    return ventanas

ventanas_c03 = time_series_cv_manual(train_c03, n_ventanas=4, horizonte=12)

print("Ventanas de validación generadas:")
for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
    print(f"Fold {i}: Train {train_fold.index.min().strftime('%Y-%m')} a {train_fold.index.max().strftime('%Y-%m')} "
          f"({len(train_fold)}m) -> Val {val_fold.index.min().strftime('%Y-%m')} a {val_fold.index.max().strftime('%Y-%m')} "
          f"({len(val_fold)}m)")

Ventanas de validación generadas:
Fold 1: Train 2018-01 a 2020-12 (36m) -> Val 2021-01 a 2021-12 (12m)
Fold 2: Train 2018-01 a 2021-12 (48m) -> Val 2022-01 a 2022-12 (12m)
Fold 3: Train 2018-01 a 2022-12 (60m) -> Val 2023-01 a 2023-12 (12m)
Fold 4: Train 2018-01 a 2023-12 (72m) -> Val 2024-01 a 2024-12 (12m)


## Holt-Winters

In [11]:
def evaluar_holt_winters(train_fold, val_fold, trend=None, seasonal=None,
                         damped_trend=False, seasonal_periods=None):
    """
    Ajusta un modelo de suavizamiento exponencial (Holt-Winters)
    con los componentes especificados.
    """
    modelo = ExponentialSmoothing(
        train_fold,
        trend=trend,
        seasonal=seasonal,
        damped_trend=damped_trend,
        seasonal_periods=seasonal_periods,
        initialization_method='estimated'
    ).fit()
    
    predicciones = modelo.forecast(len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo

In [12]:
configuraciones_hw = [
    {'trend': None, 'seasonal': None, 'damped': False, 'nombre': 'SES (Simple)'},
    {'trend': 'add', 'seasonal': None, 'damped': False, 'nombre': 'Holt lineal'},
    {'trend': 'add', 'seasonal': None, 'damped': True,  'nombre': 'Holt amortiguado'},
    {'trend': 'add', 'seasonal': 'add', 'damped': False, 'nombre': 'Holt‑Winters aditivo'},
    {'trend': 'add', 'seasonal': 'add', 'damped': True,  'nombre': 'HW aditivo amortiguado'},
]

resultados_cv_c03 = []

print("-" * 50)
print("Validación cruzada - Holt-Winters (C03)")
print("-" * 50)

for config in configuraciones_hw:
    nombre = config['nombre']
    print(f"\n--- {nombre} ---")
    
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
        pred, met, _ = evaluar_holt_winters(
            train_fold, val_fold,
            trend=config['trend'],
            seasonal=config['seasonal'],
            damped_trend=config['damped'],
            seasonal_periods=12 if config['seasonal'] else None
        )
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_c03.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_c03 = pd.DataFrame(resultados_cv_c03)
mejor_idx = df_cv_c03['RMSE'].idxmin()
mejor_nombre = df_cv_c03.loc[mejor_idx, 'modelo']
mejor_config = next(c for c in configuraciones_hw if c['nombre'] == mejor_nombre)

print("\n" + "-" * 50)
print("Mejor modelo Holt‑Winters según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_c03.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Holt-Winters (C03)
--------------------------------------------------

--- SES (Simple) ---
  Fold 1: RMSE=33.29, MAE=28.08, MAPE=94.83%, SMAPE=50.27%, MSE=1108.53
  Fold 2: RMSE=1603.21, MAE=1383.52, MAPE=258.59%, SMAPE=171.72%, MSE=2570280.09
  Fold 3: RMSE=728.88, MAE=641.25, MAPE=76.83%, SMAPE=46.25%, MSE=531262.75
  Fold 4: RMSE=708.67, MAE=628.31, MAPE=74.73%, SMAPE=48.59%, MSE=502213.04
  Promedio -> RMSE=768.51, MAE=670.29, MAPE=126.25%, SMAPE=79.21%, MSE=901216.10

--- Holt lineal ---
  Fold 1: RMSE=22.58, MAE=18.62, MAPE=55.88%, SMAPE=41.66%, MSE=509.66
  Fold 2: RMSE=1622.48, MAE=1400.03, MAPE=246.74%, SMAPE=174.84%, MSE=2632433.20
  Fold 3: RMSE=874.76, MAE=819.84, MAPE=91.70%, SMAPE=54.94%, MSE=765196.80
  Fold 4: RMSE=818.62, MAE=737.72, MAPE=86.11%, SMAPE=54.15%, MSE=670137.79
  Promedio -> RMSE=834.61, MAE=744.05, MAPE=120.11%, SMAPE=81.40%, MSE=1017069.36

--- Holt amortiguado ---
  Fold 1: RMSE=23

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
0,SES (Simple),768.51,670.29,126.25,79.21,901216.10
2,Holt amortiguado,816.54,723.51,120.97,79.17,984842.03
1,Holt lineal,834.61,744.05,120.11,81.40,1017069.36
4,HW aditivo amortiguado,835.54,743.81,152.49,98.24,976729.44
3,Holt‑Winters aditivo,858.69,770.42,141.30,107.09,1024617.61


In [13]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_c03 = ExponentialSmoothing(
    train_c03,
    trend=mejor_config['trend'],
    seasonal=mejor_config['seasonal'],
    damped_trend=mejor_config['damped'],
    seasonal_periods=12 if mejor_config['seasonal'] else None,
    initialization_method='estimated'
).fit()

pred_test_c03 = modelo_final_c03.forecast(len(test_c03))

real_test = test_c03.values
pred_test = pred_test_c03.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - SES (Simple)
--------------------------------------------------
RMSE: 499.66
MAE: 432.80
MAPE: 78.88%
SMAPE: 50.45%
MSE: 249657.59


In [14]:
meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_c03.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_c03.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_c03.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## ARIMA

In [15]:
def evaluar_arima(train_fold, val_fold, order=(1,0,0)):
    """
    Ajusta un modelo ARIMA con el orden (p,d,q) dado.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = ARIMA(train_fold, order=order)
    modelo_ajustado = modelo.fit()
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [16]:
ordenes_arima = [
    ((1,0,0), 'ARIMA(1,0,0)'),
    ((2,0,0), 'ARIMA(2,0,0)'),
    ((3,0,0), 'ARIMA(3,0,0)'),
    ((0,0,1), 'ARIMA(0,0,1)'),
    ((0,0,2), 'ARIMA(0,0,2)'),
    ((0,0,3), 'ARIMA(0,0,3)'),
    ((1,0,1), 'ARIMA(1,0,1)'),
    ((2,0,1), 'ARIMA(2,0,1)'),
    ((1,0,2), 'ARIMA(1,0,2)'),
    ((1,1,0), 'ARIMA(1,1,0)'),
    ((1,1,1), 'ARIMA(1,1,1)'),
    ((0,1,1), 'ARIMA(0,1,1)'),
    ((2,1,0), 'ARIMA(2,1,0)'),
    ((2,1,1), 'ARIMA(2,1,1)'),
]

resultados_cv_arima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos ARIMA (C03)")
print("-" * 50)

for orden, nombre in ordenes_arima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
        predicciones, met, _ = evaluar_arima(train_fold, val_fold, order=orden)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_arima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_arima = pd.DataFrame(resultados_cv_arima)
mejor_idx = df_cv_arima['RMSE'].idxmin()
mejor_orden, mejor_nombre = ordenes_arima[mejor_idx]

print("\n" + "-" * 50)
print("Mejor modelo ARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_arima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos ARIMA (C03)
--------------------------------------------------

--- ARIMA(1,0,0) ---
  Fold 1: RMSE=212.71, MAE=206.99, MAPE=560.03%, SMAPE=134.98%, MSE=45243.80
  Fold 2: RMSE=1473.59, MAE=1272.73, MAPE=433.75%, SMAPE=149.99%, MSE=2171457.58
  Fold 3: RMSE=548.23, MAE=419.50, MAPE=55.28%, SMAPE=33.53%, MSE=300555.75
  Fold 4: RMSE=516.01, MAE=429.94, MAPE=46.81%, SMAPE=37.46%, MSE=266266.72
  Promedio -> RMSE=687.63, MAE=582.29, MAPE=273.97%, SMAPE=88.99%, MSE=695880.96

--- ARIMA(2,0,0) ---
  Fold 1: RMSE=218.49, MAE=213.04, MAPE=575.66%, SMAPE=136.28%, MSE=47738.49
  Fold 2: RMSE=1474.30, MAE=1273.34, MAPE=431.40%, SMAPE=150.13%, MSE=2173555.57
  Fold 3: RMSE=545.59, MAE=409.47, MAPE=51.61%, SMAPE=33.56%, MSE=297669.75
  Fold 4: RMSE=514.29, MAE=423.07, MAPE=46.98%, SMAPE=36.58%, MSE=264495.52
  Promedio -> RMSE=688.17, MAE=579.73, MAPE=276.41%, SMAPE=89.14%, MSE=695864.83

--- ARIMA(3,0,0

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
0,"ARIMA(1,0,0)",687.63,582.29,273.97,88.99,695880.96
1,"ARIMA(2,0,0)",688.17,579.73,276.41,89.14,695864.83
2,"ARIMA(3,0,0)",688.38,578.19,251.17,88.89,710953.26
8,"ARIMA(1,0,2)",691.88,579.86,257.46,89.06,712355.38
7,"ARIMA(2,0,1)",692.49,581.56,282.66,88.80,696122.66
6,"ARIMA(1,0,1)",694.47,587.30,286.76,90.14,698131.65
9,"ARIMA(1,1,0)",767.96,671.07,118.09,75.49,903886.43
13,"ARIMA(2,1,1)",769.13,670.49,119.60,75.89,906018.61
12,"ARIMA(2,1,0)",769.90,671.58,125.92,77.39,905005.83
11,"ARIMA(0,1,1)",774.05,676.52,125.17,78.58,910230.44


In [17]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_arima = ARIMA(train_c03, order=mejor_orden)
modelo_final_arima_ajustado = modelo_final_arima.fit()

pred_test_arima = modelo_final_arima_ajustado.forecast(steps=len(test_c03))

real_test = test_c03.values
pred_test = pred_test_arima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - ARIMA(1,0,0)
--------------------------------------------------
RMSE: 440.12
MAE: 323.02
MAPE: 48.84%
SMAPE: 39.96%
MSE: 193703.73


In [18]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_arima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_arima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_arima.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## SARIMA

In [19]:
def evaluar_sarima(train_fold, val_fold, order=(1,0,0), seasonal_order=(0,0,0,12)):
    """
    Ajusta un modelo SARIMAX (SARIMA) con los órdenes especificados.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = SARIMAX(
        train_fold,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    modelo_ajustado = modelo.fit(disp=False)
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [20]:
configuraciones_sarima = [
    ((1,0,0), (0,0,0,12), 'SARIMA(1,0,0)(0,0,0)[12]'),
    ((1,0,0), (1,0,0,12), 'SARIMA(1,0,0)(1,0,0)[12]'),
    ((1,0,0), (0,0,1,12), 'SARIMA(1,0,0)(0,0,1)[12]'),
    ((1,0,0), (1,0,1,12), 'SARIMA(1,0,0)(1,0,1)[12]'),
    ((2,0,0), (0,0,0,12), 'SARIMA(2,0,0)(0,0,0)[12]'),
    ((2,0,0), (1,0,0,12), 'SARIMA(2,0,0)(1,0,0)[12]'),
    ((1,0,1), (0,0,0,12), 'SARIMA(1,0,1)(0,0,0)[12]'),
    ((1,0,1), (1,0,0,12), 'SARIMA(1,0,1)(1,0,0)[12]'),
    ((0,0,1), (0,0,0,12), 'SARIMA(0,0,1)(0,0,0)[12]'),
    ((0,0,1), (1,0,0,12), 'SARIMA(0,0,1)(1,0,0)[12]'),
    ((1,0,0), (0,1,1,12), 'SARIMA(1,0,0)(0,1,1)[12]'),
    ((0,0,1), (0,1,1,12), 'SARIMA(0,0,1)(0,1,1)[12]'),
    ((1,0,1), (0,1,1,12), 'SARIMA(1,0,1)(0,1,1)[12]'),
    ((1,1,0), (0,0,0,12), 'SARIMA(1,1,0)(0,0,0)[12]'),
    ((1,1,0), (1,0,0,12), 'SARIMA(1,1,0)(1,0,0)[12]'),
    ((0,1,1), (0,0,0,12), 'SARIMA(0,1,1)(0,0,0)[12]'),
    ((0,1,1), (0,1,1,12), 'SARIMA(0,1,1)(0,1,1)[12]'),
]

resultados_cv_sarima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos SARIMA (C03)")
print("-" * 50)

for order, seasonal_order, nombre in configuraciones_sarima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
        _, met, _ = evaluar_sarima(train_fold, val_fold, order=order, seasonal_order=seasonal_order)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_sarima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_sarima = pd.DataFrame(resultados_cv_sarima)
mejor_idx = df_cv_sarima['RMSE'].idxmin()
mejor_nombre = df_cv_sarima.loc[mejor_idx, 'modelo']
mejor_order, mejor_seasonal, _ = next((o, s, n) for o, s, n in configuraciones_sarima if n == mejor_nombre)

print("\n" + "-" * 50)
print("Mejor modelo SARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_sarima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos SARIMA (C03)
--------------------------------------------------

--- SARIMA(1,0,0)(0,0,0)[12] ---
  Fold 1: RMSE=37.74, MAE=32.23, MAPE=59.47%, SMAPE=94.30%, MSE=1424.07
  Fold 2: RMSE=1640.84, MAE=1416.43, MAPE=250.04%, SMAPE=178.24%, MSE=2692348.95
  Fold 3: RMSE=722.43, MAE=632.50, MAPE=76.10%, SMAPE=45.79%, MSE=521899.09
  Fold 4: RMSE=545.07, MAE=438.44, MAPE=52.11%, SMAPE=36.93%, MSE=297101.78
  Promedio -> RMSE=736.52, MAE=629.90, MAPE=109.43%, SMAPE=88.81%, MSE=878193.47

--- SARIMA(1,0,0)(1,0,0)[12] ---
  Fold 1: RMSE=39.60, MAE=34.66, MAPE=67.29%, SMAPE=109.25%, MSE=1568.20
  Fold 2: RMSE=1643.69, MAE=1418.84, MAPE=243.90%, SMAPE=178.69%, MSE=2701728.05
  Fold 3: RMSE=679.42, MAE=564.33, MAPE=70.83%, SMAPE=42.02%, MSE=461614.00
  Fold 4: RMSE=471.85, MAE=413.83, MAPE=44.23%, SMAPE=36.69%, MSE=222645.28
  Promedio -> RMSE=708.64, MAE=607.92, MAPE=106.56%, SMAPE=91.66%, MSE=846888.88


,modelo,RMSE,MAE,MAPE,SMAPE,MSE
5,"SARIMA(2,0,0)(1,0,0)[12]",680.40,571.10,103.05,92.70,8.124954e+05
7,"SARIMA(1,0,1)(1,0,0)[12]",690.87,580.35,130.33,84.69,8.121570e+05
6,"SARIMA(1,0,1)(0,0,0)[12]",695.69,582.78,100.64,76.54,8.242658e+05
4,"SARIMA(2,0,0)(0,0,0)[12]",696.44,582.63,101.93,81.49,8.257807e+05
1,"SARIMA(1,0,0)(1,0,0)[12]",708.64,607.92,106.56,91.66,8.468889e+05
0,"SARIMA(1,0,0)(0,0,0)[12]",736.52,629.90,109.43,88.81,8.781935e+05
14,"SARIMA(1,1,0)(1,0,0)[12]",743.33,641.04,118.94,75.08,8.667128e+05
13,"SARIMA(1,1,0)(0,0,0)[12]",767.92,671.03,118.05,75.47,9.038511e+05
15,"SARIMA(0,1,1)(0,0,0)[12]",774.62,677.19,126.46,79.31,9.104898e+05
9,"SARIMA(0,0,1)(1,0,0)[12]",812.40,713.95,132.17,121.57,9.956189e+05


In [21]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_sarima = SARIMAX(
    train_c03,
    order=mejor_order,
    seasonal_order=mejor_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
)
modelo_final_sarima_ajustado = modelo_final_sarima.fit(disp=False)

pred_test_sarima = modelo_final_sarima_ajustado.forecast(steps=len(test_c03))

real_test = test_c03.values
pred_test = pred_test_sarima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - SARIMA(2,0,0)(1,0,0)[12]
--------------------------------------------------
RMSE: 421.36
MAE: 271.51
MAPE: 31.97%
SMAPE: 34.16%
MSE: 177542.77


In [22]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_sarima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_sarima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_sarima.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Dynamic Optimized Theta

In [23]:
def evaluar_theta(train_fold, val_fold, theta=2.0, period=12):
    """
    Ajusta un modelo Theta con el parámetro theta dado.
    Procedimiento:
      1. Descomponer la serie con STL (period=12) para obtener estacionalidad.
      2. Serie desestacionalizada = serie - estacionalidad.
      3. Ajustar regresión lineal a la serie desestacionalizada (tendencia global).
      4. Ajustar SES a la serie desestacionalizada.
      5. Combinar pronósticos: Yhat = SES_forecast + theta*(trend_forecast - SES_forecast).
      6. Sumar estacionalidad (último año).
    Retorna predicciones, métricas y un diccionario con información del modelo.
    """
    if len(train_fold) < 2 * period:
        ses = ExponentialSmoothing(train_fold, trend=None, seasonal=None,
                                   initialization_method='estimated').fit()
        pred = ses.forecast(len(val_fold))
        real = val_fold.values
        pred = pred.values
        mse = mean_squared_error(real, pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(real, pred)
        mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
        smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
        return pred, {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}, {'theta': theta, 'method': 'SES'}
    
    stl = STL(train_fold, period=period, robust=True).fit()
    seasonal = stl.seasonal
    deseasonalized = train_fold - seasonal

    x = np.arange(len(deseasonalized))
    slope, intercept, _, _, _ = sp_stats.linregress(x, deseasonalized.values)
    trend_line = slope * x + intercept

    ses_model = ExponentialSmoothing(deseasonalized, trend=None, seasonal=None,
                                     initialization_method='estimated').fit()
    
    h = len(val_fold)
    ses_forecast = ses_model.forecast(h)
    last_idx = len(deseasonalized) - 1
    trend_extrapolated = slope * np.arange(last_idx + 1, last_idx + 1 + h) + intercept

    theta_forecast = ses_forecast.values + theta * (trend_extrapolated - ses_forecast.values)

    last_year_seasonal = seasonal.iloc[-period:].values
    repeats = int(np.ceil(h / period))
    seasonal_pattern = np.tile(last_year_seasonal, repeats)[:h]
    final_forecast = theta_forecast + seasonal_pattern

    real = val_fold.values
    pred = final_forecast

    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}
    modelo_info = {'theta': theta, 'slope': slope, 'intercept': intercept}
    
    return final_forecast, metricas, modelo_info

In [24]:
thetas = np.arange(0.0, 4.0 + 0.25, 0.25)

resultados_cv_theta = []

print("-" * 50)
print("Validación cruzada - Optimización de Theta (C03)")
print("-" * 50)

for theta in thetas:
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
        _, met, _ = evaluar_theta(train_fold, val_fold, theta=theta, period=12)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_theta.append({
        'theta': theta,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })

df_cv_theta = pd.DataFrame(resultados_cv_theta)
mejor_idx = df_cv_theta['RMSE'].idxmin()
mejor_theta = df_cv_theta.loc[mejor_idx, 'theta']
mejor_rmse_cv = df_cv_theta.loc[mejor_idx, 'RMSE']

print("\n--- Resultados de optimización de theta ---")
print(f"Mejor theta: {mejor_theta:.2f} (RMSE CV promedio = {mejor_rmse_cv:.2f})")
print("\nTop 5 configuraciones:")
display(df_cv_theta.sort_values('RMSE').head())

print("\n" + "-" * 50)
print(f"Validación cruzada - Theta dinámico óptimo (θ={mejor_theta:.2f})")
print("-" * 50)

metricas_opt_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
    _, met, _ = evaluar_theta(train_fold, val_fold, theta=mejor_theta, period=12)
    metricas_opt_folds['RMSE'].append(met['RMSE'])
    metricas_opt_folds['MAE'].append(met['MAE'])
    metricas_opt_folds['MAPE'].append(met['MAPE'])
    metricas_opt_folds['SMAPE'].append(met['SMAPE'])
    metricas_opt_folds['MSE'].append(met['MSE'])
    print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
          f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")

prom_opt = {k: np.mean(v) for k, v in metricas_opt_folds.items()}
print(f"  Promedio -> RMSE={prom_opt['RMSE']:.2f}, MAE={prom_opt['MAE']:.2f}, "
      f"MAPE={prom_opt['MAPE']:.2f}%, SMAPE={prom_opt['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Optimización de Theta (C03)
--------------------------------------------------

--- Resultados de optimización de theta ---
Mejor theta: 0.75 (RMSE CV promedio = 730.63)

Top 5 configuraciones:


,theta,RMSE,MAE,MAPE,SMAPE,MSE
3,0.75,730.629977,617.264459,186.131795,100.936713,842378.660509
4,1.00,737.546693,604.561187,186.796090,100.880786,857412.237521
2,0.50,765.517269,655.997397,188.099619,100.143429,879579.953475
5,1.25,790.987194,642.034003,193.817571,102.461959,924680.684510
1,0.25,820.564273,717.082322,193.822696,102.176257,969016.116419



--------------------------------------------------
Validación cruzada - Theta dinámico óptimo (θ=0.75)
--------------------------------------------------
  Fold 1: RMSE=210.09, MAE=153.82, MAPE=442.80%, SMAPE=142.49%, MSE=44137.03
  Fold 2: RMSE=1666.99, MAE=1441.36, MAPE=218.39%, SMAPE=192.01%, MSE=2778844.49
  Fold 3: RMSE=517.46, MAE=444.59, MAPE=39.01%, SMAPE=34.55%, MSE=267765.39
  Fold 4: RMSE=527.98, MAE=429.28, MAPE=44.34%, SMAPE=34.70%, MSE=278767.73
  Promedio -> RMSE=730.63, MAE=617.26, MAPE=186.13%, SMAPE=100.94%, MSE=5112135.02


In [25]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - Theta optimizado (θ={mejor_theta:.2f})")
print("-" * 50)

pred_test_theta, met_train, modelo_info = evaluar_theta(train_c03, test_c03, theta=mejor_theta, period=12)

real_test = test_c03.values
pred_test = pred_test_theta

mse_test = met_train['MSE']
rmse_test = met_train['RMSE']
mae_test = met_train['MAE']
mape_test = met_train['MAPE']
smape_test = met_train['SMAPE']

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c03.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Theta optimizado (θ=0.75)
--------------------------------------------------
RMSE: 679.18
MAE: 582.47
MAPE: 102.04%
SMAPE: 61.29%
MSE: 461282.50


In [26]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_theta,
    mode='lines+markers', name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - Theta Dinámico Optimizado (θ={mejor_theta:.2f}): Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_theta,
    mode='lines+markers', name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_theta

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Ridge

In [27]:
def crear_dataset_supervisado_ridge(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_ridge(train_fold, val_fold, n_input=12, alphas=None):
    """
    Ajusta RidgeCV con características determinísticas.
    Retorna predicciones, métricas, modelo y scaler.
    """
    if alphas is None:
        alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
    
    X_train, y_train = crear_dataset_supervisado_ridge(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = RidgeCV(alphas=alphas, cv=tscv)
    modelo.fit(X_train_scaled, y_train)
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [28]:
resultados_cv_ridge = []

print("-" * 50)
print("Validación cruzada - Ridge Regression (C03)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_ridge(train_fold, val_fold)
    
    resultados_cv_ridge.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.2f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_ridge = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_ridge]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_ridge]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_ridge]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_ridge]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_ridge])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_ridge['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_ridge['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_ridge['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_ridge['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_ridge['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Ridge Regression (C03)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 10.00
RMSE: 267.87
MAE: 240.39
MAPE: 640.23%
SMAPE: 136.30%
MSE: 71755.63

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 1000.00
RMSE: 1533.77
MAE: 1336.17
MAPE: 927.45%
SMAPE: 157.20%
MSE: 2352437.96

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 1000.00
RMSE: 1084.78
MAE: 997.38
MAPE: 79.02%
SMAPE: 141.00%
MSE: 1176737.78

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 1000.00
RMSE: 664.62
MAE: 545.25
MAPE: 43.28%
SMAPE: 59.19%
MSE: 441725.11

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 887.76
MAE: 779.80
MAPE: 422.50%
SMAPE: 123.42%
MSE: 1010664.1

In [29]:
print("-" * 50)
print("Evaluación final en Test (2025) - Ridge Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_ridge(
    train_c03.values, train_c03.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

tscv = TimeSeriesSplit(n_splits=3)
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
modelo_final_ridge = RidgeCV(alphas=alphas, cv=tscv)
modelo_final_ridge.fit(X_train_full_scaled, y_train_full)
print(f"Alpha seleccionado: {modelo_final_ridge.alpha_:.2f}")

x_lags_pred_test = train_c03.values[-12:]
fecha_fin_pred_test = train_c03.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_ridge = modelo_final_ridge.predict(X_pred_test_scaled).flatten()

real_test = test_c03.values
pred_test = pred_test_ridge

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c03.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Ridge Regression
--------------------------------------------------
Alpha seleccionado: 1000.00
RMSE: 410.50
MAE: 312.55
MAPE: 50.91%
SMAPE: 38.90%
MSE: 168510.11


In [30]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - Ridge Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_ridge

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Lasso

In [31]:
def crear_dataset_supervisado_lasso(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lasso(train_fold, val_fold, n_input=12, alphas=None):
    """
    Ajusta MultiTaskLassoCV con características determinísticas.
    Retorna predicciones, métricas, modelo y scaler.
    """
    if alphas is None:
        alphas = np.logspace(-4, 4, 50)
    
    X_train, y_train = crear_dataset_supervisado_lasso(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = MultiTaskLassoCV(alphas=alphas, cv=tscv, max_iter=20000, random_state=42)
    modelo.fit(X_train_scaled, y_train)
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [32]:
resultados_cv_lasso = []

print("-" * 50)
print("Validación cruzada - Lasso Regression (C03)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lasso(train_fold, val_fold)
    
    resultados_cv_lasso.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.4f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_lasso = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lasso]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lasso]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lasso]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lasso]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lasso])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lasso['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lasso['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lasso['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lasso['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lasso['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Lasso Regression (C03)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 10000.0000
RMSE: 268.45
MAE: 243.97
MAPE: 621.91%
SMAPE: 138.42%
MSE: 72062.78

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 10000.0000
RMSE: 1527.59
MAE: 1331.79
MAPE: 967.41%
SMAPE: 156.07%
MSE: 2333537.85

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 10000.0000
RMSE: 908.40
MAE: 851.00
MAPE: 69.95%
SMAPE: 110.22%
MSE: 825193.00

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 339.3222
RMSE: 507.55
MAE: 433.14
MAPE: 52.06%
SMAPE: 37.24%
MSE: 257604.67

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 803.00
MAE: 714.98
MAPE: 427.83%
SMAPE: 110.49%
MSE:

In [33]:
print("-" * 50)
print("Evaluación final en Test (2025) - Lasso Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lasso(
    train_c03.values, train_c03.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

tscv = TimeSeriesSplit(n_splits=3)
alphas = np.logspace(-4, 4, 50)
modelo_final_lasso = MultiTaskLassoCV(alphas=alphas, cv=tscv, max_iter=20000, random_state=42)
modelo_final_lasso.fit(X_train_full_scaled, y_train_full)
print(f"Alpha seleccionado: {modelo_final_lasso.alpha_:.2f}")

x_lags_pred_test = train_c03.values[-12:]
fecha_fin_pred_test = train_c03.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_lasso = modelo_final_lasso.predict(X_pred_test_scaled).flatten()

real_test = test_c03.values
pred_test = pred_test_lasso

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c03.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Lasso Regression
--------------------------------------------------
Alpha seleccionado: 494.17
RMSE: 665.53
MAE: 619.67
MAPE: 115.48%
SMAPE: 64.02%
MSE: 442936.80


In [34]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - Lasso Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_lasso

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Random Forest

In [35]:
def crear_dataset_supervisado_rf(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_random_forest(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta Random Forest multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [5, 10, 15, None],
            'estimator__min_samples_split': [2, 5, 10]
        }
    
    X_train, y_train = crear_dataset_supervisado_rf(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    rf = RandomForestRegressor(random_state=42)
    multi_rf = MultiOutputRegressor(rf)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_rf, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [36]:
resultados_cv_rf = []

print("-" * 50)
print("Validación cruzada - Random Forest (C03)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_random_forest(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    min_samples_split = best_params['estimator__min_samples_split']
    
    resultados_cv_rf.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, min_samples_split={min_samples_split}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_rf = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_rf]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_rf]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_rf]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_rf]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_rf])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_rf['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_rf['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_rf['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_rf['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_rf['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Random Forest (C03)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=5, min_samples_split=10
RMSE: 264.74
MAE: 244.33
MAPE: 627.67%
SMAPE: 139.23%
MSE: 70084.89

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=200, max_depth=5, min_samples_split=10
RMSE: 1556.93
MAE: 1357.19
MAPE: 643.31%
SMAPE: 167.54%
MSE: 2424035.48

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=200, max_depth=5, min_samples_split=10
RMSE: 666.17
MAE: 578.71
MAPE: 50.67%
SMAPE: 50.32%
MSE: 443778.13

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=5, min_samples_split=10
RMSE: 648.42
MAE: 551.64
MAPE: 65.11%
SMAPE: 44.06%
MSE: 420448.81

--------------------------------------

In [37]:
print("-" * 50)
print("Evaluación final en Test (2025) - Random Forest")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_rf(
    train_c03.values, train_c03.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [5, 10, 15, None],
    'estimator__min_samples_split': [2, 5, 10]
}

rf_final = RandomForestRegressor(random_state=42)
multi_rf_final = MultiOutputRegressor(rf_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_rf_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_rf = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c03.values[-12:]
fecha_fin_pred_test = train_c03.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_rf = modelo_final_rf.predict(X_pred_test_scaled).flatten()

real_test = test_c03.values
pred_test = pred_test_rf

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c03.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Random Forest
--------------------------------------------------
Mejores parámetros: {'estimator__max_depth': 15, 'estimator__min_samples_split': 2, 'estimator__n_estimators': 400}
RMSE: 519.62
MAE: 472.24
MAPE: 87.90%
SMAPE: 53.39%
MSE: 270006.41


In [38]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - Random Forest: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_rf

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## XGBoost

In [39]:
def crear_dataset_supervisado_xgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_xgboost(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta XGBoost multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_xgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    xgb = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
    multi_xgb = MultiOutputRegressor(xgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_xgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [40]:
resultados_cv_xgb = []

print("-" * 50)
print("Validación cruzada - XGBoost (C03)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_xgboost(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_xgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_xgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_xgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_xgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_xgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_xgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_xgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_xgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_xgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_xgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_xgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_xgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - XGBoost (C03)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=1.0, colsample_bytree=0.8
RMSE: 285.10
MAE: 271.94
MAPE: 704.46%
SMAPE: 145.56%
MSE: 81280.16

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 1543.35
MAE: 1348.90
MAPE: 754.85%
SMAPE: 163.81%
MSE: 2381938.16

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=7, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 525.25
MAE: 439.20
MAPE: 38.53%
SMAPE: 53.18%
MSE: 275885.89

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=200, max_depth=3, learning_rate=0.01, subsampl

In [41]:
print("-" * 50)
print("Evaluación final en Test (2025) - XGBoost")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_xgb(
    train_c03.values, train_c03.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, 9],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

xgb_final = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
multi_xgb_final = MultiOutputRegressor(xgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_xgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_xgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c03.values[-12:]
fecha_fin_pred_test = train_c03.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_xgb = modelo_final_xgb.predict(X_pred_test_scaled).flatten()

real_test = test_c03.values
pred_test = pred_test_xgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c03.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - XGBoost
--------------------------------------------------
Mejores parámetros: {'estimator__colsample_bytree': 0.8, 'estimator__learning_rate': 0.05, 'estimator__max_depth': 7, 'estimator__n_estimators': 300, 'estimator__subsample': 0.8}
RMSE: 516.97
MAE: 471.37
MAPE: 82.00%
SMAPE: 52.98%
MSE: 267262.36


In [42]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - XGBoost: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_xgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## LightGBM

In [43]:
def crear_dataset_supervisado_lgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lightgbm(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta LightGBM multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7, -1],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_lgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    lgb = LGBMRegressor(random_state=42, verbosity=-1)
    multi_lgb = MultiOutputRegressor(lgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_lgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [44]:
resultados_cv_lgb = []

print("-" * 50)
print("Validación cruzada - LightGBM (C03)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lightgbm(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_lgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_lgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - LightGBM (C03)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 268.45
MAE: 243.97
MAPE: 621.91%
SMAPE: 138.42%
MSE: 72062.78

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 1527.59
MAE: 1331.79
MAPE: 967.41%
SMAPE: 156.07%
MSE: 2333537.85

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 908.40
MAE: 851.00
MAPE: 69.95%
SMAPE: 110.22%
MSE: 825193.00

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsam

In [45]:
print("-" * 50)
print("Evaluación final en Test (2025) - LightGBM")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lgb(
    train_c03.values, train_c03.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, -1],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

lgb_final = LGBMRegressor(random_state=42, verbosity=-1)
multi_lgb_final = MultiOutputRegressor(lgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_lgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_lgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c03.values[-12:]
fecha_fin_pred_test = train_c03.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_lgb = modelo_final_lgb.predict(X_pred_test_scaled).flatten()

real_test = test_c03.values
pred_test = pred_test_lgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c03.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - LightGBM
--------------------------------------------------
Mejores parámetros: {'estimator__colsample_bytree': 1.0, 'estimator__learning_rate': 0.01, 'estimator__max_depth': 3, 'estimator__n_estimators': 100, 'estimator__subsample': 0.8}
RMSE: 536.22
MAE: 480.50
MAPE: 87.96%
SMAPE: 54.38%
MSE: 287527.45


In [46]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - LightGBM: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_lgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## KNN

In [47]:
def crear_dataset_supervisado_knn(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_knn(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta KNN multi‑output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_neighbors': [2, 3, 5, 7, 10],
            'estimator__weights': ['uniform', 'distance'],
            'estimator__p': [1, 2]
        }
    
    X_train, y_train = crear_dataset_supervisado_knn(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    knn = KNeighborsRegressor()
    multi_knn = MultiOutputRegressor(knn)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_knn, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [48]:
resultados_cv_knn = []

print("-" * 50)
print("Validación cruzada - KNN Regression (C03)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c03, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_knn(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_neighbors = best_params['estimator__n_neighbors']
    weights = best_params['estimator__weights']
    p = best_params['estimator__p']
    
    resultados_cv_knn.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_neighbors': n_neighbors,
        'weights': weights,
        'p': p,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_neighbors={n_neighbors}, weights={weights}, p={p}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_knn = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_knn]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_knn]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_knn]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_knn]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_knn])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_knn['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_knn['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_knn['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_knn['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_knn['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - KNN Regression (C03)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_neighbors=2, weights=distance, p=2
RMSE: 397.05
MAE: 343.46
MAPE: 903.73%
SMAPE: 146.45%
MSE: 157649.68

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_neighbors=7, weights=distance, p=2
RMSE: 1617.70
MAE: 1404.80
MAPE: 523.98%
SMAPE: 176.75%
MSE: 2616962.32

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_neighbors=10, weights=uniform, p=1
RMSE: 1030.32
MAE: 942.94
MAPE: 74.18%
SMAPE: 126.10%
MSE: 1061558.40

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_neighbors=7, weights=distance, p=1
RMSE: 730.52
MAE: 618.01
MAPE: 50.24%
SMAPE: 71.47%
MSE: 533655.32

--------------------------------------------------
Métricas promedio en validación cruzada
--

In [49]:
print("-" * 50)
print("Evaluación final en Test (2025) - KNN Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_knn(
    train_c03.values, train_c03.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_neighbors': [2, 3, 5, 7, 10, 12],
    'estimator__weights': ['uniform', 'distance'],
    'estimator__p': [1, 2]
}

knn_final = KNeighborsRegressor()
multi_knn_final = MultiOutputRegressor(knn_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_knn_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_knn = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c03.values[-12:]
fecha_fin_pred_test = train_c03.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_knn = modelo_final_knn.predict(X_pred_test_scaled).flatten()

real_test = test_c03.values
pred_test = pred_test_knn

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c03.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - KNN Regression
--------------------------------------------------
Mejores parámetros: {'estimator__n_neighbors': 10, 'estimator__p': 1, 'estimator__weights': 'uniform'}
RMSE: 443.21
MAE: 369.73
MAPE: 65.25%
SMAPE: 44.74%
MSE: 196435.76


In [50]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c03.index, y=train_c03.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=test_c03.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c03.index, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C03 - KNN Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c03.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c03.values - pred_test_knn

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Tabla 1 – Métricas Promedio en Validación Cruzada (CV)

| Modelo                     | RMSE    | MAE     | MAPE     | SMAPE   | MSE         |
|----------------------------|---------|---------|----------|---------|-------------|
| Holt‑Winters (SES)         | 768.51  | 670.29  | 126.25%  | 79.21%  | 901216.10   |
| ARIMA(1,0,0)               | 687.63  | 582.29  | 273.97%  | 88.99%  | 695880.96   |
| SARIMA(2,0,0)(1,0,0)[12]   | 680.40  | 571.10  | 103.05%  | 92.70%  | 812495.44   |
| Theta (θ=0.75)             | 730.63  | 617.26  | 186.13%  | 100.94% | 842378.66   |
| Ridge                      | 887.76  | 779.80  | 422.50%  | 123.42% | 1010664.12  |
| Lasso                      | 803.00  | 714.98  | 427.83%  | 110.49% | 872099.58   |
| Random Forest              | 784.06  | 682.97  | 346.69%  | 100.29% | 839586.82   |
| XGBoost                    | 696.88  | 609.51  | 383.77%  | 99.48%  | 731829.80   |
| LightGBM                   | 784.23  | 686.82  | 421.30%  | 108.60% | 854458.57   |
| KNN                        | 943.90  | 827.30  | 388.04%  | 130.19% | 1092456.43  |

## Tabla 2 – Métricas en Test (2025)

| Modelo                     | RMSE    | MAE     | MAPE    | SMAPE   | MSE         |
|----------------------------|---------|---------|---------|---------|-------------|
| Holt‑Winters (SES)         | 499.66  | 432.80  | 78.88%  | 50.45%  | 249657.59   |
| ARIMA(1,0,0)               | 440.12  | 323.02  | 48.84%  | 39.96%  | 193703.73   |
| SARIMA(2,0,0)(1,0,0)[12]   | 421.36  | 271.51  | 31.97%  | 34.16%  | 177542.77   |
| Theta (θ=0.75)             | 679.18  | 582.47  | 102.04% | 61.29%  | 461282.50   |
| Ridge                      | 410.50  | 312.55  | 50.91%  | 38.90%  | 168510.11   |
| Lasso                      | 665.53  | 619.67  | 115.48% | 64.02%  | 442936.80   |
| Random Forest              | 519.62  | 472.24  | 87.90%  | 53.39%  | 270006.41   |
| XGBoost                    | 516.97  | 471.37  | 82.00%  | 52.98%  | 267262.36   |
| LightGBM                   | 536.22  | 480.50  | 87.96%  | 54.38%  | 287527.45   |
| KNN                        | 443.21  | 369.73  | 65.25%  | 44.74%  | 196435.76   |

## Tabla 3 – Errores por Mes en Test (2025)

| Modelo                     | Ene  | Feb  | Mar  | Abr  | May  | Jun  | Jul  | Ago  | Sep   | Oct  | Nov  | Dic  |
|----------------------------|------|------|------|------|------|------|------|------|-------|------|------|------|
| Holt‑Winters (SES)         | -292 | -367 | -546 | 66   | -504 | -709 | -409 | -753 | -284  | 887  | -23  | -364 |
| ARIMA(1,0,0)               | -186 | -215 | -355 | 289  | -253 | -435 | -116 | -444 | 38    | 1210 | 320  | -13  |
| SARIMA(2,0,0)(1,0,0)[12]   | 118  | -18  | -161 | 328  | -210 | -190 | 91   | -160 | -18   | 1254 | 501  | 209  |
| Theta (θ=0.75)             | 351  | -117 | -517 | -294 | -841 | -720 | -601 | -700 | -1414 | 284  | -238 | -912 |
| Ridge                      | -55  | -128 | -300 | 300  | -286 | -489 | -186 | -523 | -82   | 1063 | 160  | -179 |
| Lasso                      | -395 | -481 | -684 | -133 | -753 | -971 | -697 | -1039| -624  | 501  | -410 | -748 |
| Random Forest              | -266 | -452 | -497 | 10   | -576 | -704 | -636 | -865 | -234  | 525  | -426 | -476 |
| XGBoost                    | -92  | -606 | -480 | -116 | -659 | -740 | -507 | -591 | -227  | 733  | -474 | -431 |
| LightGBM                   | -394 | -434 | -616 | -26  | -577 | -777 | -479 | -786 | -347  | 792  | -110 | -430 |
| KNN                        | -141 | -224 | -351 | 200  | -407 | -642 | -327 | -660 | -192  | 944  | 85   | -264 |

## Interpretación de los Resultados de los Modelos

**a) Desempeño en validación cruzada (CV, 2018‑2024)**  
- Los modelos SARIMA(2,0,0)(1,0,0)[12] y ARIMA(1,0,0) mostraron los menores RMSE promedio (680 y 688 respectivamente), destacándose en el ajuste histórico.  
- Las métricas MAPE y SMAPE están muy elevadas (≥79 %) en todos los modelos, lo cual se debe a que en ciertos meses del período de validación (especialmente el Fold 1) los valores reales fueron muy bajos, magnificando los errores porcentuales.  
- Modelos de aprendizaje automático (XGBoost, Random Forest, LightGBM) tuvieron RMSE similares o ligeramente superiores a los ARIMA, sin lograr ventaja clara en CV.

**b) Desempeño en el conjunto de prueba (2025)**  
- El mejor RMSE lo obtuvo Ridge (410.50), seguido de SARIMA (421.36), ARIMA (440.12), KNN (443.21) y Holt‑Winters SES (499.66).  
- Todos los modelos cometieron un error grave en octubre de 2025, subestimando el valor real en más de 1000 comparendos (el error más bajo fue de +733 en XGBoost y el más alto +1254 en SARIMA). Esto indica un pico atípico no capturado por ningún modelo.  
- Los modelos sin componente de tendencia (SES, ARIMA sin deriva) mostraron un sesgo creciente hacia la subestimación en los últimos meses del año, mientras que Ridge y KNN, que pueden incorporar tendencia, tuvieron errores más equilibrados (salvo octubre).  
- A pesar del pico de octubre, el desempeño global de Ridge fue el más robusto, con un MAE de 312 y MAPE de 50.9 %, superando a los demás métodos clásicos y de machine learning.

**c) Patrones de error mensual en 2025**  
- Octubre concentra el mayor error positivo para todos los modelos (real mucho mayor que el pronóstico).  
- Noviembre y diciembre muestran subestimación en SARIMA y ARIMA, mientras que Ridge y Holt‑Winters logran errores menores en esos meses.  
- Los meses de mediados de año (junio, julio, agosto) son generalmente sobrestimados por los modelos (errores negativos) debido a la alta variabilidad estacional no explicada.

---

## ¿Por qué se llegó a estos resultados?

**a) Naturaleza de la serie (código C03 – Bloqueo de calzada)**  
- La descomposición STL evidenció una tendencia positiva significativa (+10.3 comparendos/mes, R²=0.515, p<0.0001). Esto significa que la infracción por bloqueo de calzada viene creciendo de forma sostenida.  
- La serie original es estacionaria en tendencia (ADF p=0.0196), es decir, su comportamiento es estable alrededor de una tendencia determinista.  
- La estacionalidad es moderada (10.9 % de la varianza), y los residuos presentan autocorrelación fuerte en los primeros rezagos (Ljung‑Box significativo), además de no ser normales ni centrados en cero. Esto indica que el modelo STL no eliminó por completo la dependencia temporal.

**b) Consecuencias para los modelos**  
- Los modelos sin tendencia explícita (Holt‑Winters SES, ARIMA(1,0,0) sin deriva, SARIMA(2,0,0)(1,0,0)[12] sin intercepto variable) asumen que la serie fluctúa alrededor de una media constante. Al existir una tendencia creciente clara, estos modelos subestiman sistemáticamente los valores a medida que avanza el tiempo, lo que se manifestó en el test de 2025 con errores positivos grandes en los últimos meses.  
- Ridge, en cambio, incluye como variable regresora un índice temporal que captura el crecimiento lineal. Esta es la razón principal de su mejor desempeño en el período de prueba: extrapola la tendencia ascendente, reduciendo el sesgo acumulado.  
- El pico extremo de octubre de 2025 no fue anticipado por ningún modelo porque no existía un precedente similar en el historial; todos fallaron por igual en ese mes.  
- Los modelos de árboles (Random Forest, XGBoost, LightGBM) no lograron superar a Ridge porque, aunque pueden capturar relaciones no lineales, la estructura dominante en esta serie es una tendencia lineal y una estacionalidad débil, que una regresión lineal regularizada modela de forma más controlada y estable.

**c) Desempeño en validación vs. prueba**  
- En CV, los modelos ARIMA y SARIMA fueron los mejores porque el proceso de validación corta los datos antes de que la tendencia cause un sesgo acumulado severo. Pero al evaluar en el futuro (2025), la tendencia penaliza a esos modelos, haciendo que Ridge destaque como la opción más realista para el pronóstico a largo plazo.

---

## Modelo Ideal para Predecir 2026

Se recomienda utilizar la regresión regularizada Ridge (con tendencia y estacionalidad) como modelo de pronóstico para 2026.

**Justificación:**  
- Fue el modelo con menor RMSE en el test de 2025 (410.5) y errores más equilibrados fuera del evento atípico de octubre.  
- Incorpora explícitamente la tendencia positiva que caracteriza a la infracción C03, a diferencia de los modelos ARIMA estacionarios o el suavizamiento exponencial simple, que subestimarían el volumen futuro.  
- Ridge logra un balance entre ajuste y regularización, lo que lo hace robusto ante la alta variabilidad y los valores extremos presentes en la serie.  
- Su pronóstico para 2026 reflejará la continuación del crecimiento observado, con las fluctuaciones estacionales habituales.

## Pronóstico de la Infracción C03 para el Año 2026

Una vez seleccionado el mejor modelo predictivo (Ridge Regression con validación cruzada temporal y características derivadas: rezagos de 12 meses, tendencia normalizada, codificación cíclica del mes y variable indicadora de COVID-19), se procede a realizar el pronóstico del volumen de infracciones por bloqueo de calzada o intersección para los 12 meses del año 2026. El modelo se entrena con la serie completa de 96 meses (enero 2018 a diciembre 2025) utilizando un enfoque supervisado donde la entrada son los últimos 12 meses (rezagos) y la salida son los siguientes 12 meses, empleando un escalado estándar de características. Se selecciona el hiperparámetro de regularización alpha mediante TimeSeriesSplit con 3 pliegues, y con el modelo final se predicen los 12 meses de 2026 a partir de los últimos 12 valores observados. Los resultados se visualizan mediante dos gráficos interactivos: el primero muestra la serie histórica completa junto con la proyección hacia 2026, con una línea vertical discontinua que marca el inicio del período pronosticado; el segundo presenta exclusivamente el pronóstico mensual para 2026, facilitando la interpretación de la tendencia esperada mes a mes. Esta visualización permite anticipar el comportamiento futuro de la infracción y apoyar la toma de decisiones en materia de control de bloqueo de calzadas e intersecciones.

In [51]:
def crear_dataset_supervisado_ridge(serie_valores, fechas, n_input=12, n_output=12):
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)


serie_vals = serie_c03.values
fechas = serie_c03.index

X_full, y_full = crear_dataset_supervisado_ridge(serie_vals, fechas, n_input=12, n_output=12)

scaler_final = StandardScaler()
X_full_scaled = scaler_final.fit_transform(X_full)

tscv = TimeSeriesSplit(n_splits=3)
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
modelo_ridge_2026 = RidgeCV(alphas=alphas, cv=tscv)
modelo_ridge_2026.fit(X_full_scaled, y_full)


ultimos_12 = serie_vals[-12:] 

fecha_fin_input = pd.Timestamp('2025-12-01')
mes_fin = fecha_fin_input.month  
mes_sin = np.sin(2 * np.pi * mes_fin / 12) 
mes_cos = np.cos(2 * np.pi * mes_fin / 12)  
tendencia = 1.0  
covid = 0        

x_pred_2026 = np.concatenate([ultimos_12, [tendencia, mes_sin, mes_cos, covid]])
X_pred_2026_scaled = scaler_final.transform(x_pred_2026.reshape(1, -1))

forecast_2026 = modelo_ridge_2026.predict(X_pred_2026_scaled).flatten()

idx_forecast = pd.date_range(start='2026-01-01', periods=12, freq='MS')
forecast_series = pd.Series(forecast_2026, index=idx_forecast)

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=serie_c03.index,
    y=serie_c03.values,
    mode='lines+markers',
    name='Histórico (2018-2025)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=forecast_series.index,
    y=forecast_series.values,
    mode='lines+markers',
    name=f'Predicción Ridge (α={modelo_ridge_2026.alpha_:.2f})',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2026-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2026-01-01',
    y=1,
    yref='paper',
    text='<b>Inicio 2026</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text='C03 - Ridge Regression: Predicción 2026<br>'
             '<sup>Modelo entrenado con datos hasta diciembre 2025</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()

meses_2026 = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
              'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses_2026,
    y=forecast_2026,
    mode='lines+markers',
    name=f'Predicción Ridge (α={modelo_ridge_2026.alpha_:.2f})',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Pronóstico C03 para el año 2026',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig2.show()